### import bibliotek


In [47]:
import os
import requests
from bs4 import BeautifulSoup
import json

### wysyłanie żądania dostępu do strony z opiniami oprodukcie

In [41]:
url = "https://www.ceneo.pl/84514582#tab=reviews"
response = requests.get(url)
response.status_code


200

In [45]:
with open("./headers.json", "r") as jf:
    headers = json.load(jf)

### Dla każdej z opini wydobycie z kodu  HTML poszczególnych składowych i zapisanie ich w postaci złożonych struktur danych

In [54]:
product_id = "178891349;20136-0v"
next_page = f"https://www.ceneo.pl/{product_id}#tab=reviews"
all_reviews = []
while next_page:
    response = requests.get(next_page, headers=headers)
    print(next_page)
    if response.status_code == 200:
       page_dom = BeautifulSoup(response.text, "html.parser")
       reviews = page_dom.select("div.js_product-review:not(.user-post--highlight)")
       print(len(reviews))
    for review in reviews:
        try:
            single_review = {
                "review_id": review['data-entry-id'],
                "author": review.select_one("span.user-post__author-name").text.strip(),
                "recomendation": review.select_one("span.user-post__author-recomendation > em").text.strip(),
                "stars": review.select_one("span.user-post__score-count").text.strip(),
                "pros": [p.text.strip() for p in review.select("div.review-feature__item--positive")],
                "cons": [c.text.strip() for c in review.select("div.review-feature__item--negative")],
                "content": review.select_one("div.user-post__text").text.strip(),
                "likes": review.select_one("button.vote-yes > span").text.strip(),
                "dislikes" : review.select_one("button.vote-no > span").text.strip(),
                "publish_date": review.select_one("span.user-post__published > time:nth-child(1)")['datetime'].strip(),
                "publish_date": review.select_one("span.user-post__published > time:nth-child(2)")['datetime'].strip(),
            }  
            all_reviews.append(single_review)
            
        except (AttributeError, TypeError):
            pass 
    try:
        next_page = "https://www.ceneo.pl"+page_dom.select_one("a.pagination__next")["href"]
    except TypeError:
        next_page = None    
        
               

https://www.ceneo.pl/178891349;20136-0v#tab=reviews
0


### Zapisywanie wszystkich opini o konkretnym produkcie w bazie danych

In [50]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [51]:
with open(f"./opinions/{product_id}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_reviews, jf, indent=4, ensure_ascii=False)